# 🎯 [Colab 실습] YOLO v1 첫걸음 — 순수 파이썬으로 밑바닥부터

**온디바이스 AI 프로그래밍 · Day 3 「YOLOv9 실시간 파이프라인」 들어가기 전 준비 실습**

| 항목 | 내용 |
| --- | --- |
| 대상 | YOLO가 "박스 그려주는 마법 상자"로만 느껴지는 분 |
| 도구 | **순수 파이썬 + numpy + matplotlib만** — PyTorch 없음! |
| 환경 | Google Colab CPU 런타임 (학습 셀 ☕ 3~8분 1회 포함) |
| 진행 | 위에서부터 셀을 하나씩 실행 (`Shift + Enter`) |

## 이 실습의 목표

2016년 YOLO v1 논문의 심장 세 가지 — **그리드 분할 · 책임 셀 · 3항 손실** — 를 직접 구현해,
**미니 YOLO를 밑바닥부터 학습시켜 실제로 물체를 검출**합니다. 마지막에는 우리 미니 YOLO가
논문 v1의 유명한 약점("작은 물체에 약하다")까지 **그대로 재현**하는 것을 확인하게 됩니다.

## 로드맵

| Part | 주제 | 핵심 발견 |
| --- | --- | --- |
| 1 | "무엇"에서 "무엇이 어디에"로 | 검출 문제와 슬라이딩 윈도우의 비효율 |
| 2 | YOLO의 발상 — 그리드와 책임 셀 | 출력 텐서 S×S×(5+C), 좌표 인코딩 |
| 3 | 손실 함수 — 세 가지 책임의 저울 | λ_coord=5, λ_noobj=0.5의 이유 |
| 4 | 백본 — conv가 필요한 이유 | MLP 실패 실험 → im2col conv |
| 5 | 🏁 학습과 검출의 순간 | 검출 성공 + 작은 물체의 저주 재현 |
| 6 | 정리 — 진짜 v1, 그리고 v9까지 | Day 3 실습으로 가는 지도 |

> 💡 각 Step의 **✅ 확인**을 점검하고 **✏️ 직접 해보기**로 실험하세요.


---
# Part 0. 환경 준비

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)
rng = np.random.default_rng(42)

try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
    print("한글 폰트 설정 완료")
except Exception as e:
    print("한글 폰트 생략:", e)
plt.rc("axes", unicode_minus=False)
print("준비 완료 🎯")

---
# Part 1. "무엇"에서 "무엇이 어디에"로

### Step 1-1. 미니 검출 세계 만들기

지금까지의 실습에서 우리는 4×4 이미지가 "0인가 1인가 7인가"만 물었습니다(분류).
이제 무대를 넓힙니다: **16×16 이미지 어딘가에** 숫자 스탬프가 **작게(4px) 또는 크게(8px)** 찍혀 있습니다.
질문이 바뀝니다 — **"무엇이, 어디에, 얼마나 크게 있는가?"** 답의 형식은 **박스**: (클래스, cx, cy, w, h).

In [ ]:
IMG, S, C = 16, 4, 3          # 이미지 16×16, 그리드 4×4, 클래스 3종
CELL = IMG // S                # 셀 한 변 = 4px
D = 5 + C                      # 셀당 출력: [x,y,w,h,conf] + 클래스 3 = 8

B0 = np.array([[1,1,1,1],[1,0,0,1],[1,0,0,1],[1,1,1,1]], float)
B1 = np.array([[0,0,1,0],[0,1,1,0],[0,0,1,0],[0,1,1,1]], float)
B7 = np.array([[1,1,1,1],[0,0,0,1],[0,0,1,0],[0,1,0,0]], float)
BASES = [B0, B1, B7]
NAMES = ["0", "1", "7"]

def make_sample(noise=0.18):
    """물체 1개를 임의 위치·크기(4px/8px)로 배치. 반환: 이미지, 정답박스"""
    img = rng.normal(0, noise, (IMG, IMG))
    cls = int(rng.integers(0, 3))
    scale = int(rng.choice((1, 2)))                    # 1→4px, 2→8px
    sz = 4 * scale
    x0 = int(rng.integers(0, IMG - sz + 1))
    y0 = int(rng.integers(0, IMG - sz + 1))
    img[y0:y0+sz, x0:x0+sz] += np.kron(BASES[cls], np.ones((scale, scale)))
    return img, (cls, x0 + sz/2, y0 + sz/2, sz, sz)    # (클래스, 중심x, 중심y, 폭, 높이)

def draw_box(ax, box, color, label=None):
    cls, cx, cy, w, h = box[:5]
    ax.add_patch(patches.Rectangle((cx-w/2-0.5, cy-h/2-0.5), w, h,
                 fill=False, edgecolor=color, lw=2))
    if label: ax.text(cx-w/2-0.5, cy-h/2-1.1, label, color=color, fontsize=9, weight="bold")

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax in axes:
    img, gt = make_sample()
    ax.imshow(img, cmap="gray"); ax.axis("off")
    draw_box(ax, gt, "#2ca02c", f"{NAMES[gt[0]]} @({gt[1]:.0f},{gt[2]:.0f}) {gt[3]:.0f}px")
plt.suptitle("미니 검출 세계 — 정답 = 클래스 + 박스(cx,cy,w,h)"); plt.tight_layout(); plt.show()

### Step 1-2. 옛날 방식의 대가 — 슬라이딩 윈도우

YOLO 이전의 상식: **창(window)을 밀며 모든 위치·크기를 잘라 분류기에 넣는다**.
우리 미니 세계에서 비용을 계산해 봅시다.

In [ ]:
n_windows = 0
for sz in (4, 8):                                     # 크기 2종
    n_pos = (IMG - sz + 1) ** 2                       # 가능한 위치
    n_windows += n_pos
    print(f"창 {sz}×{sz}: 위치 {IMG-sz+1}×{IMG-sz+1} = {n_pos}곳")
print(f"\n→ 이미지 '한 장'을 처리하는 데 분류기 추론 {n_windows}회!")
print(f"   (실전 해상도·크기·비율이면 수만~수십만 회 — R-CNN 계열이 느렸던 이유)")
print()
print("💡 YOLO의 질문: \"이미지를 '한 번만 보고(You Only Look Once)'")
print("   모든 위치의 답을 동시에 내놓을 수는 없을까?\"")

> **✅ Part 1 확인**
> - [ ] 검출의 답 형식이 (클래스, cx, cy, w, h)임을 안다
> - [ ] 슬라이딩 윈도우가 추론 수백 회를 요구함을 계산했다

---
# Part 2. YOLO의 발상 — 그리드와 책임 셀

### Step 2-1. 이미지를 S×S로 나누고, 셀에게 책임을 묻는다

YOLO v1의 한 문장 요약:

> **"이미지를 S×S 그리드로 나누고, 물체의 '중심'이 떨어진 셀이 그 물체를 책임지고 예측한다."**

우리는 S=4 (논문은 S=7). 책임 셀은 나눗셈 한 번으로 정해집니다.

In [ ]:
def responsible_cell(cx, cy):
    """물체 중심 (cx,cy)가 속한 셀 (행 i, 열 j)"""
    j = min(int(cx // CELL), S - 1)     # 열 = x 방향
    i = min(int(cy // CELL), S - 1)     # 행 = y 방향
    return i, j

img, gt = make_sample()
cls, cx, cy, w, h = gt
i, j = responsible_cell(cx, cy)

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.imshow(img, cmap="gray")
for k in range(1, S):                                  # 그리드 선
    ax.axhline(k*CELL - 0.5, color="#4c72b0", lw=0.8, alpha=0.7)
    ax.axvline(k*CELL - 0.5, color="#4c72b0", lw=0.8, alpha=0.7)
ax.add_patch(patches.Rectangle((j*CELL-0.5, i*CELL-0.5), CELL, CELL,
             fill=True, facecolor="#e8a33d", alpha=0.35))
ax.plot(cx-0.5, cy-0.5, "r*", ms=14)
draw_box(ax, gt, "#2ca02c")
ax.set_title(f"중심(★)이 셀 (행{i}, 열{j})에 → 이 셀이 책임 셀"); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"물체 '{NAMES[cls]}' 중심 ({cx:.1f}, {cy:.1f}) → 책임 셀 = (행 {i}, 열 {j})")

### Step 2-2. 출력 텐서 설계 — 이미지 한 장 = 텐서 하나

각 셀이 내놓을 답: 박스 하나 `[x, y, w, h, conf]` + 클래스 확률 `[p0, p1, p7]` = **8개 숫자**.
셀이 16개이므로 네트워크의 출력은 **4×4×8 텐서** 하나 — 이것이 "한 번만 본다"의 실체입니다.

$$\text{출력} = S \times S \times (B \cdot 5 + C) \quad \Big|\quad \text{우리: } 4{\times}4{\times}8 \;(B{=}1) \quad \text{논문: } 7{\times}7{\times}30 \;(B{=}2, C{=}20)$$

**conf(idence)** 의 의미: "이 셀에 물체가 있고, 내 박스가 정확할 자신감" — 논문 정의는 Pr(Object)×IoU.

### Step 2-3. 좌표 인코딩 — 셀의 언어로 번역하기

픽셀 좌표를 그대로 회귀시키지 않고 **셀 기준으로 번역**합니다 (학습이 훨씬 안정적):

- **(tx, ty)**: 중심의 **셀 내 상대 위치** (0~1) — 셀이 바뀌어도 같은 스케일
- **(√w̄, √h̄)**: 이미지 대비 비율의 **제곱근** — 큰 박스의 오차가 손실을 독점하지 못하게 (v1의 유명한 트릭, Part 5에서 재조명)

In [ ]:
def encode_target(box):
    """정답 박스 → 4×4×8 타깃 텐서 (책임 셀만 채움)"""
    cls, cx, cy, w, h = box
    T = np.zeros((S, S, D))
    i, j = responsible_cell(cx, cy)
    tx = (cx - j*CELL) / CELL              # 셀 내 상대 x (0~1)
    ty = (cy - i*CELL) / CELL
    T[i, j] = [tx, ty, np.sqrt(w/IMG), np.sqrt(h/IMG), 1.0, *np.eye(3)[cls]]
    return T

def decode_cell(p_cell, i, j):
    """셀 (i,j)의 8칸 벡터 → 픽셀 좌표 박스 (인코딩의 역연산)"""
    tx, ty, sw, sh = p_cell[:4]
    cx, cy = (j + tx) * CELL, (i + ty) * CELL
    w, h = max(sw, 0)**2 * IMG, max(sh, 0)**2 * IMG
    return int(np.argmax(p_cell[5:])), cx, cy, w, h, p_cell[4]

T = encode_target(gt)
print(f"타깃 텐서 shape: {T.shape} — 채워진 셀: {int((T[...,4]==1).sum())}개 (책임 셀뿐)")
print(f"책임 셀 (행{i},열{j})의 벡터: {np.round(T[i,j], 3)}")
print(f"        [tx, ty, √w̄, √h̄, conf, p0, p1, p7]")

# 왕복 검증: encode → decode = 원본
back = decode_cell(T[i, j], i, j)
print(f"\n디코딩 복원: 클래스 {NAMES[back[0]]}, 중심 ({back[1]:.1f},{back[2]:.1f}), 크기 {back[3]:.1f}px")
print(f"원본과 일치: {np.allclose([back[1],back[2],back[3]], [cx,cy,w])}")

> **✅ Part 2 확인**
> - [ ] 책임 셀 = 중심 좌표 나눗셈 한 번임을 확인했다
> - [ ] 출력 4×4×8 (논문 7×7×30)의 구성 요소를 나열할 수 있다
> - [ ] encode↔decode 왕복이 원본을 복원함을 검증했다

> 📎 함께 보기: HTML 애니메이션 「YOLO v1 책임 셀」 — 슬라이더로 물체를 옮기며 책임 셀·(tx,ty)가 변하는 것을 봅니다.

---
# Part 3. 손실 함수 — 세 가지 책임의 저울

### Step 3-1. 세 항의 분업

v1 손실은 전부 **제곱합**이고, 셀의 역할에 따라 세 항으로 나뉩니다:

$$L = \lambda_{coord}\!\!\sum_{책임셀}\!\big[(x{-}\hat x)^2{+}(y{-}\hat y)^2{+}(\sqrt w{-}\sqrt{\hat w})^2{+}(\sqrt h{-}\sqrt{\hat h})^2\big] + \sum_{책임셀}(1{-}\hat{c})^2 + \lambda_{noobj}\!\!\sum_{빈셀}\hat{c}^2 + \sum_{책임셀}\sum_k(p_k{-}\hat p_k)^2$$

| 항 | 누가 | 목표 | 가중치 |
| --- | --- | --- | --- |
| 좌표 | 책임 셀만 | 박스를 정답에 | **λ_coord = 5** |
| conf | 책임 셀 / 빈 셀 | 1로 / **0으로** | 1 / **λ_noobj = 0.5** |
| 클래스 | 책임 셀만 | one-hot에 | 1 |

### Step 3-2. λ는 왜 5와 0.5인가 — 침묵하는 다수의 문제

In [ ]:
n_obj, n_noobj = 1, S*S - 1
print(f"셀 16개 중 책임 셀 {n_obj}개 vs 빈 셀 {n_noobj}개 — 빈 셀이 {n_noobj}배 많다!")
print()
print("가중치 없이 학습하면 생기는 일:")
print(f"  빈 셀 {n_noobj}개의 'conf→0' 신호가 책임 셀 1개의 신호를 압도")
print(f"  → 네트워크의 최적해: '모든 conf를 0으로, 좌표는 대충' (검출 포기)")
print()
print("v1의 처방:")
print(f"  λ_noobj=0.5 → 빈 셀 목소리를 절반으로  |  λ_coord=5 → 박스 정확도 목소리를 5배로")
print(f"  체감 비율: 좌표 5 : conf(책임) 1 : conf(빈셀) {0.5*n_noobj:.1f} — 저울이 맞음")

### Step 3-3. 구현 — 제곱합의 선물

제곱합 손실의 기울기는 `2·(pred − target)`. 마스크(책임/빈)와 λ만 곱하면 끝이라
**역전파 출발점(dP)이 세 줄**로 구현됩니다.

In [ ]:
L_COORD, L_NOOBJ = 5.0, 0.5

def yolo_loss_and_grad(P, T):
    """P, T: (N, 4, 4, 8). 반환: 스칼라 손실, dL/dP"""
    obj = T[..., 4:5]                                          # 책임 셀 마스크 (N,4,4,1)
    dP = np.zeros_like(P)
    dP[..., 0:4] = 2 * L_COORD * obj * (P[..., 0:4] - T[..., 0:4])            # ① 좌표
    dP[..., 4:5] = 2 * (obj * (P[..., 4:5] - 1)
                        + L_NOOBJ * (1 - obj) * P[..., 4:5])                  # ② conf
    dP[..., 5:]  = 2 * obj * (P[..., 5:] - T[..., 5:])                        # ③ 클래스
    loss = (L_COORD * obj * (P[..., 0:4] - T[..., 0:4])**2).sum() \
         + (obj * (P[..., 4:5] - 1)**2 + L_NOOBJ * (1-obj) * P[..., 4:5]**2).sum() \
         + (obj * (P[..., 5:] - T[..., 5:])**2).sum()
    return loss / len(P), dP / len(P)

P_rand = rng.normal(0, 0.1, (2, S, S, D))
T_demo = np.stack([encode_target(make_sample()[1]) for _ in range(2)])
loss, dP = yolo_loss_and_grad(P_rand, T_demo)
print(f"무작위 예측의 손실: {loss:.3f}")
print(f"dP shape {dP.shape} — 책임 셀 외 좌표·클래스 기울기가 0인지: "
      f"{np.all(dP[T_demo[...,4]==0][:, :4]==0) and np.all(dP[T_demo[...,4]==0][:, 5:]==0)}")

> **✅ Part 3 확인**
> - [ ] 세 항(좌표/conf/클래스)의 담당 셀과 목표를 표로 그릴 수 있다
> - [ ] λ_coord=5, λ_noobj=0.5가 "1 vs 15" 불균형의 저울임을 이해했다
> - [ ] 제곱합 덕에 기울기가 세 줄임을 확인했다

---
# Part 4. 백본 — conv가 필요한 이유 (실패 실험 포함)

### Step 4-1. 데이터 준비 (실행만 하면 됩니다)

In [ ]:
def make_set(n):
    X, T, L = [], [], []
    for _ in range(n):
        img, box = make_sample()
        X.append(img); T.append(encode_target(box)); L.append(box)
    return np.array(X), np.array(T), L

Xtr, Ttr, Ltr = make_set(2000)
Xte, Tte, Lte = make_set(200)
print(f"학습 {len(Xtr)}장 / 테스트 {len(Xte)}장 준비 완료")

### Step 4-2. 순진한 시도 — flatten + MLP (짧은 실패 실험, ~30초)

이미지를 한 줄로 펴서 2층 MLP에 넣어봅니다. 결과를 보고 이유를 생각해 봅시다.

In [ ]:
def iou(a, b):
    """두 박스 (cls,cx,cy,w,h,...)의 IoU"""
    ax1,ay1,ax2,ay2 = a[1]-a[3]/2, a[2]-a[4]/2, a[1]+a[3]/2, a[2]+a[4]/2
    bx1,by1,bx2,by2 = b[1]-b[3]/2, b[2]-b[4]/2, b[1]+b[3]/2, b[2]+b[4]/2
    ix = max(0, min(ax2,bx2) - max(ax1,bx1)); iy = max(0, min(ay2,by2) - max(ay1,by1))
    inter = ix * iy
    return inter / (a[3]*a[4] + b[3]*b[4] - inter + 1e-9)

def decode_best(p):
    """conf가 가장 높은 셀 하나를 박스로"""
    conf = p[..., 4]
    i, j = np.unravel_index(conf.argmax(), conf.shape)
    return decode_cell(p[i, j], i, j)

def evaluate(P_test):
    ious, ok = [], 0
    for k in range(len(P_test)):
        pred = decode_best(P_test[k]); gt = Lte[k]
        ious.append(iou(pred, gt)); ok += (pred[0] == gt[0])
    return np.mean(ious), np.mean(np.array(ious) > 0.5)*100, ok/len(P_test)*100

# 2층 MLP: 256 → 96 → 128
Xf_tr, Xf_te = Xtr.reshape(2000, -1), Xte.reshape(200, -1)
r = np.random.default_rng(7)
Wm1 = r.normal(0, np.sqrt(2/256), (256, 96)); bm1 = np.zeros(96)
Wm2 = r.normal(0, np.sqrt(2/96), (96, 128)); bm2 = np.zeros(128)
lr = 0.05
for ep in range(400):
    h = np.maximum(0, Xf_tr @ Wm1 + bm1)
    P = (h @ Wm2 + bm2).reshape(-1, S, S, D)
    _, dP = yolo_loss_and_grad(P, Ttr)
    d = dP.reshape(2000, -1)
    Wm2 -= lr*(h.T @ d); bm2 -= lr*d.sum(0)
    dh = d @ Wm2.T * (h > 0)
    Wm1 -= lr*(Xf_tr.T @ dh); bm1 -= lr*dh.sum(0)

P_mlp = (np.maximum(0, Xf_te @ Wm1 + bm1) @ Wm2 + bm2).reshape(-1, S, S, D)
iou_mlp, pct_mlp, acc_mlp = evaluate(P_mlp)
print(f"MLP 결과: 평균 IoU {iou_mlp:.3f} | IoU>0.5 비율 {pct_mlp:.0f}% | 클래스 {acc_mlp:.0f}%   ← 벽에 부딪힘")
print()
print("💡 MLP도 2000장을 통째로 외우는 힘으로 어느 정도는 따라옵니다. 하지만 flatten이")
print("   '위치'를 지워버려 — 좌상단의 7과 우하단의 7이 완전히 다른 입력이 되어 —")
print("   위치마다 처음부터 다시 배워야 하고, 검출률이 낮은 곳에서 멈춥니다.")
print("   conv를 쓰면 같은 학습량으로 얼마나 오르는지 Part 5에서 직접 비교합니다.")

### Step 4-3. conv의 약속 — 어디서든 같은 특징

conv 필터는 **같은 가중치로 모든 위치를 훑습니다**(weight sharing). 좌상단에서 배운 "7의 모서리"가
우하단에서도 그대로 통합니다 — 검출에 필요한 **위치 등변성**입니다.

구현은 「가상 NPU」 실습의 그 무기를 재사용합니다: **conv = im2col + GEMM**.
(첫 층이라 입력 쪽 기울기가 필요 없어 역전파도 GEMM 하나입니다!)

In [ ]:
K_SZ, STRIDE, OUT = 4, 2, 7            # 4×4 필터, 보폭 2 → 특징맵 7×7

def im2col(X):
    """(N,16,16) → (N·49, 16): 각 창을 행으로 펼침 (가상 NPU 실습과 동일 원리)"""
    N = len(X)
    col = np.empty((N, OUT, OUT, K_SZ*K_SZ))
    for oy in range(OUT):
        for ox in range(OUT):
            col[:, oy, ox, :] = X[:, oy*STRIDE:oy*STRIDE+K_SZ,
                                     ox*STRIDE:ox*STRIDE+K_SZ].reshape(N, -1)
    return col.reshape(N*OUT*OUT, -1)

K, H = 16, 160                          # 필터 16개, FC 은닉 160
r = np.random.default_rng(7)
Wc = r.normal(0, np.sqrt(2/16), (16, K));      bc = np.zeros(K)
W1 = r.normal(0, np.sqrt(2/(49*K)), (49*K, H)); b1 = np.zeros(H)
W2 = r.normal(0, np.sqrt(2/H), (H, S*S*D));     b2 = np.zeros(S*S*D)

Xcol_tr, Xcol_te = im2col(Xtr), im2col(Xte)    # 미리 펼쳐두기 (입력 고정이므로 1회)

def forward(Xcol, N):
    F = np.maximum(0, Xcol @ Wc + bc)          # conv (GEMM!) + ReLU → (N·49, K)
    f = F.reshape(N, -1)                       # 특징맵 flatten (N, 49K)
    h = np.maximum(0, f @ W1 + b1)             # FC + ReLU
    return (h @ W2 + b2).reshape(N, S, S, D), h, f, F

n_params = Wc.size + W1.size + W2.size + K + H + S*S*D
print(f"미니 YOLO: conv(4×4,s2,K={K}) → FC{H} → 출력 4×4×8  |  파라미터 {n_params:,}개")
print("📎 v1도 마지막은 FC였습니다: conv 24층 → FC 4096 → 'FC로' 7×7×30 출력")

> **✅ Part 4 확인**
> - [ ] MLP가 위치 일반화에 실패하는 것(IoU ≈ 0.1)을 목격했다
> - [ ] conv = im2col + GEMM으로 백본을 만들었다 (가상 NPU 실습 연결)
> - [ ] "v1의 출력도 FC"라는 사실을 알았다

---
# Part 5. 🏁 학습과 검출의 순간

### Step 5-1. 학습 — 손실 3항으로 2200 에폭 (☕ 3~8분, 1회만 실행)

In [ ]:
import time
lr = 0.05
losses = []
t0 = time.time()
for ep in range(1, 2201):
    if ep in (1300, 1800): lr *= 0.5                    # 계단식 감쇠
    P, h, f, F = forward(Xcol_tr, 2000)
    loss, dP = yolo_loss_and_grad(P, Ttr)
    losses.append(loss)
    d = dP.reshape(2000, -1)
    gW2 = h.T @ d;  gb2 = d.sum(0)
    dh = d @ W2.T * (h > 0)
    gW1 = f.T @ dh; gb1 = dh.sum(0)
    dF = (dh @ W1.T).reshape(2000*49, K) * (F > 0)      # conv 역전파도 GEMM 하나
    Wc -= lr*(Xcol_tr.T @ dF); bc -= lr*dF.sum(0)
    W2 -= lr*gW2; b2 -= lr*gb2; W1 -= lr*gW1; b1 -= lr*gb1
    if ep % 400 == 0:
        print(f"  ep {ep:>4}: loss {loss:.4f}  ({time.time()-t0:.0f}s)")

plt.figure(figsize=(7, 2.8))
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("YOLO 손실")
plt.title("3항 손실이 함께 내려간다"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Step 5-2. 검출의 순간 — 박스가 그려진다

In [ ]:
P_te, _, _, _ = forward(Xcol_te, 200)

fig, axes = plt.subplots(2, 3, figsize=(10.5, 7))
for ax, k in zip(axes.flat, range(6)):
    ax.imshow(Xte[k], cmap="gray"); ax.axis("off")
    gt = Lte[k]; pred = decode_best(P_te[k])
    draw_box(ax, gt, "#2ca02c", f"정답 {NAMES[gt[0]]}")
    draw_box(ax, pred, "#e8a33d", f"예측 {NAMES[pred[0]]} conf {pred[5]:.2f}")
    ax.set_title(f"IoU = {iou(pred, gt):.2f}", fontsize=10)
plt.suptitle("초록=정답 · 주황=미니 YOLO의 예측"); plt.tight_layout(); plt.show()

### Step 5-3. 정량 평가

In [ ]:
m_iou, pct, acc = evaluate(P_te)
cerr = np.mean([np.hypot(decode_best(P_te[k])[1]-Lte[k][1],
                          decode_best(P_te[k])[2]-Lte[k][2]) for k in range(200)])
obj_conf = P_te[Tte[..., 4] == 1][:, 4].mean()
noobj_conf = P_te[Tte[..., 4] == 0][:, 4].mean()

print("════ 미니 YOLO 성적표 (테스트 200장) ════")
print(f"평균 IoU        : {m_iou:.3f}")
print(f"IoU > 0.5 비율  : {pct:.0f}%   (검출 성공 기준)")
print(f"클래스 정확도    : {acc:.0f}%")
print(f"중심 오차 평균   : {cerr:.2f}px  ← 1픽셀 미만!")
print(f"conf 분리       : 책임 셀 {obj_conf:.2f} vs 빈 셀 {noobj_conf:.2f}  ← λ 저울이 작동한 증거")
print()
print(f"MLP(4-2) 대비   : IoU {iou_mlp:.3f}→{m_iou:.3f}, 검출률 {pct_mlp:.0f}%→{pct:.0f}%, 클래스 {acc_mlp:.0f}%→{acc:.0f}%")
print(f"                 — 같은 손실·같은 데이터에서 conv(위치 등변성)가 만든 차이")

### Step 5-4. 작은 물체의 저주 — 논문 v1의 약점을 재현하다

전체 수치는 준수한데, **크기별로 쪼개 보면** 흥미로운 일이 벌어집니다.

In [ ]:
from collections import defaultdict
by_size = defaultdict(list); w_pred = defaultdict(list)
for k in range(200):
    pred = decode_best(P_te[k]); gt = Lte[k]
    by_size[gt[3]].append(iou(pred, gt)); w_pred[gt[3]].append(pred[3])

print(f"{'물체 크기':<10}{'평균 IoU':>10}{'IoU>0.5':>10}{'예측 w 평균':>12}")
for sz in sorted(by_size):
    arr = np.array(by_size[sz])
    print(f"{int(sz)}px{'':<7}{arr.mean():>10.3f}{(arr>0.5).mean()*100:>9.0f}%{np.mean(w_pred[sz]):>11.2f}px")
print()
print("👀 크기 예측은 둘 다 정확합니다(4px→~4.2, 8px→~8.2). 그런데 4px 물체의 IoU만 무너집니다.")
print()
print("이유는 산수입니다: 4×4 박스는 중심이 1px만 어긋나도 IoU가 0.6으로,")
print("1.5px 어긋나면 0.45로 추락합니다 (8×8 박스는 같은 오차에 훨씬 관대).")
print()
print("📜 논문 v1의 고백과 정확히 같습니다: 'our model struggles with small objects' —")
print("   √w 트릭도 이 문제를 '완화'할 뿐 해결하진 못했고, v2부터 앵커 박스·고해상도로 진화한 배경입니다.")
print("   우리 미니 YOLO가 2016년 논문의 약점까지 충실히 재현한 셈입니다!")

### Step 5-5. NMS 맛보기 — 겹친 예측 정리하기

지금까진 conf 최고 셀 하나만 디코딩했지만, 실전은 **임계값을 넘는 모든 셀**을 후보로 씁니다.
이웃 셀이 같은 물체를 중복 예측하는 문제 → **NMS(비최대 억제)**: conf순 정렬 후, 채택한 박스와
많이 겹치는(IoU > 0.5) 후보를 제거.

In [ ]:
def detect_all(p, conf_th=0.3, iou_th=0.5):
    """임계값 검출 + NMS"""
    cands = []
    for i in range(S):
        for j in range(S):
            if p[i, j, 4] > conf_th:
                cands.append(decode_cell(p[i, j], i, j))
    cands.sort(key=lambda b: -b[5])                    # conf 내림차순
    kept = []
    for c in cands:
        if all(iou(c, k) <= iou_th for k in kept):     # 기존 채택과 안 겹치면
            kept.append(c)
    return cands, kept

# 이웃 셀이 중복 예측한(억제가 실제 일어나는) 샘플을 자동으로 찾아 시연
k_demo = next((k for k in range(200)
               if len(detect_all(P_te[k], conf_th=0.2)[0]) > len(detect_all(P_te[k], conf_th=0.2)[1])), 0)
cands, kept = detect_all(P_te[k_demo], conf_th=0.2)
print(f"테스트 {k_demo}번: conf>0.2 후보 {len(cands)}개 → NMS 후 {len(kept)}개")
for c in cands:
    mark = "✔ 채택" if any(c is kk for kk in kept) else "✂ 억제 (채택 박스와 IoU>0.5)"
    print(f"  {NAMES[c[0]]} @({c[1]:4.1f},{c[2]:4.1f}) conf {c[5]:.2f}  {mark}")
print()
print("📎 Day 3에서 이 NMS가 YOLOv9 후처리(post_process.py)의 핵심으로 다시 등장합니다 —")
print("   「NMS·YOLO 후처리 해부」 실습에서 실전 규모로 확장하세요.")

### Step 5-6. 리포트 과제 — 직접 실험하고 표를 채우세요

| 실험 | 조건 | 평균 IoU | 관찰 |
| --- | --- | --- | --- |
| 기준 | 위 설정 그대로 | | |
| 실험1 | √ 제거: w/IMG를 그대로 회귀 | | |
| 실험2 | λ_noobj = 5.0 (빈 셀 목소리 10배) | | |
| 실험3 | S=2 (셀 8px — 굵은 그리드) | | |

**분석 질문 (2~3문장씩):**
1. 실험1에서 어느 크기의 물체가 더 손해를 보나요? √의 역할을 "큰 박스의 손실 독점"으로 설명하세요.
2. 실험2에서 conf 분리(책임 vs 빈)와 검출률이 어떻게 변하나요? λ 저울의 의미와 연결하세요.
3. 실험3에서 "한 셀에 물체 중심 2개"가 생길 확률이 커집니다. v1이 구조적으로 이 경우를 처리 못 하는 이유는? (힌트: 셀당 클래스는 1벌)

In [ ]:
# ✏️ 실험 공간 — 재학습이 필요한 실험은 Part 4-3부터 변수만 바꿔 다시 실행하세요.
print("실험은 위 셀들을 복사해 조건만 바꿔 진행하세요!")

---
# Part 6. 정리 — 미니 YOLO ↔ 진짜 YOLO v1, 그리고 v9까지

| 오늘 만든 미니 YOLO | 논문 YOLO v1 (2016) |
| --- | --- |
| 16×16 입력, S=4 그리드 | 448×448 입력, S=7 그리드 |
| 셀당 박스 B=1 → 4×4×**8** | B=2, C=20 → 7×7×**30** |
| 책임 셀 = 중심이 속한 셀 | 동일 + B=2 중 IoU 높은 predictor가 책임 |
| conv 1층 + FC (im2col GEMM) | conv 24층(ImageNet 사전학습) + FC 4096 |
| 3항 제곱합, λ=5/0.5, √w | 완전히 동일 (우리가 그대로 구현) |
| conf 임계 + NMS | 동일 |
| **작은 물체의 저주 (스케일별 격차)** | **논문의 고백 그대로** — v2 앵커 탄생의 배경 |

## v1 → v9 한 줄 계보 (Day 3 예고)

- **v1**: 그리드+책임 셀+3항 손실 (오늘의 전부) → **v2**: 앵커 박스·고해상도 → **v3**: 멀티스케일 예측(작은 물체 처방!)
- → **v4/v5**: 백본·증강 현대화 → **v8/v9**: anchor-free(v1 정신으로 회귀!), PGI·GELAN
- 변하지 않은 것: **"한 번만 보고, 그리드가 답한다"** — 오늘 배운 뼈대가 Day 3 `post_process.py`의
  출력 텐서 해석에 그대로 살아 있습니다.

## NPU와의 연결 (교안 Day 3)

- Day 3 목표 "YOLOv9 포팅 + 20 FPS 이상": NPU는 오늘 우리가 짠 **conv(=GEMM)부터 출력 텐서 생성까지**를
  가속하고, **NMS는 CPU에서 분리 실행**됩니다 (교안: "NMS 비최대억제 후처리 분리 실행") —
  오늘의 10줄 NMS가 바로 그 CPU 쪽 일입니다.
- 모델 카탈로그의 YOLOv9(원본 24.4MB → 경량화 5.8MB)가 본 과정 기본 모델 —
  그 출력 텐서를 읽는 눈이 오늘 이미 생겼습니다.

## ✏️ 심화 도전 과제 (선택)

1. **B=2 구현**: 셀당 박스 2개(4×4×13), "IoU 높은 predictor가 책임" 규칙 추가 — v1 완전체
2. **물체 2개 세계**: make_sample이 서로 다른 셀에 물체 2개를 배치하도록 확장, NMS 포함 검출 성능 측정
3. **mAP 계산기**: conf 임계를 스윕하며 precision-recall 곡선을 그리고 AP 계산 — Day 5 평가 지표 예습
4. **트리플 콤보 배포**: 이 미니 YOLO에 「양자화 첫걸음」 INT8 + 「프루닝 첫걸음」 연쇄수술 적용 —
   검출 IoU가 몇 %p 떨어지는지 측정 ("Detection·Segmentation은 QAT 권장"이라는 교안 문장의 체험)

---

수고하셨습니다! 🎉 이제 YOLO는 "박스 그려주는 마법"이 아니라
**"그리드 셀들이 각자 책임지고 8개 숫자를 회귀하는 것"**입니다.
Day 3에서 만나는 YOLOv9의 출력 텐서도, NMS도 — 오늘 손으로 만든 그것의 확장판입니다.
